> ## MODEL v3 (CLIP ViT-B-32 frozen) -- pembanding, TIDAK live di backend
>
> Backend tetap pakai model v2. Demo ini HANYA utk presentasi/perbandingan --
> lihat `notebooks/README.md` & `notebooks/04_compare_models.ipynb`.
>
> **Scope demo lebih kecil dari `v2_22class/03_demo_inference.ipynb`**:
> bagian live webcam (section 6 di v2) TIDAK dibuat ulang di sini -- itu
> fitur UX utk model yang benar-benar dipakai user (v1/v2), sedangkan v3 di
> sini murni baseline akademik utk perbandingan metodologi. Scan dari
> file/gambar tetap ada (section 3-4 di bawah), cukup utk demo "lihat
> bedanya" di presentasi. Prasyarat: `01_train.ipynb` (classifier head) dan
> `02_embedding_retrieval.ipynb` (index katalog) di folder ini sudah dijalankan.

# 03 — Demo Inference -- MODEL v3 (CLIP frozen + linear probe)

`scan_lukisan(image_path)`: embed gambar via CLIP (frozen) -> (opsional)
prediksi style via classifier head yang dilatih di `01_train.ipynb` -> cari
tetangga terdekat di index katalog CLIP (`02_embedding_retrieval.ipynb`).

## 1. Setup

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

_p = Path.cwd()
while not ((_p / "src").is_dir() and (_p / "configs").is_dir()):
    if _p == _p.parent:
        raise RuntimeError("folder ml-visual-search/ (berisi src/ & configs/) tidak ketemu")
    _p = _p.parent
os.chdir(_p)
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
import open_clip

print("working dir:", os.getcwd())
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Muat CLIP + classifier head + katalog

In [ ]:
from src.utils import load_config, get_device, resolve_path
from src.embedding import nearest_neighbors, confidence_verdict

cfg = load_config("configs/config.yaml")
clip_cfg = cfg["baseline_clip"]
device = get_device(cfg["device"])

CKPT_PATH = "checkpoints/v3_clip/best.pth"
CATALOG_PATH = clip_cfg["embedding_output_path"]

if not resolve_path(CKPT_PATH).exists():
    raise FileNotFoundError(f"{CKPT_PATH} belum ada -- jalankan dulu 01_train.ipynb di folder ini.")
if not resolve_path(CATALOG_PATH).exists():
    raise FileNotFoundError(f"{CATALOG_PATH} belum ada -- jalankan dulu 02_embedding_retrieval.ipynb (section 5).")

ck = torch.load(resolve_path(CKPT_PATH), map_location=str(device), weights_only=False)
if "feat_mean" not in ck:
    raise RuntimeError("checkpoint v3 format LAMA (tanpa statistik standardisasi) -- "
                       "jalankan ulang 01_train.ipynb dulu.")
label_to_idx = ck["label_to_idx"]
idx_to_label = {v: k for k, v in label_to_idx.items()}
num_classes = len(label_to_idx)

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    ck["clip_model_name"], pretrained=ck["clip_pretrained"]
)
clip_model = clip_model.to(device).eval()
for p in clip_model.parameters():
    p.requires_grad_(False)

# arsitektur HARUS identik dgn build_classifier() di 01_train.ipynb section 4
clf_cfg = ck["classifier_config"]
embed_dim = ck["embed_dim"]
if clf_cfg["hidden_dim"] and clf_cfg["hidden_dim"] > 0:
    classifier = nn.Sequential(
        nn.Dropout(clf_cfg.get("input_dropout", 0.0)),
        nn.Linear(embed_dim, clf_cfg["hidden_dim"]), nn.GELU(),
        nn.Dropout(clf_cfg["dropout"]),
        nn.Linear(clf_cfg["hidden_dim"], num_classes),
    )
else:
    classifier = nn.Sequential(nn.Dropout(clf_cfg.get("input_dropout", 0.0)),
                               nn.Linear(embed_dim, num_classes))
classifier.load_state_dict(ck["classifier_state_dict"])
classifier = classifier.to(device).eval()

feat_mean = torch.tensor(ck["feat_mean"], device=device)
feat_std  = torch.tensor(ck["feat_std"], device=device)
print(f"[MODEL v3 -- CLIP {ck['clip_model_name']}] classifier head loaded "
      f"| val_macro_f1={ck['val_metrics']['macro_f1']:.4f}")

cat = np.load(resolve_path(CATALOG_PATH))
cat_emb, cat_fnames, cat_labels = cat["embeddings"], cat["filenames"], cat["labels"]
meta = pd.read_csv(resolve_path("data/raw/metadata.csv")).set_index("filename")
print(f"[KATALOG v3 CLIP] {len(cat_fnames)} karya x {cat_emb.shape[1]}-d embedding")

## 3. Fungsi `scan_lukisan(image_path)` + `show_result(result)`

In [ ]:
def scan_lukisan(image_path, top_k=5):
    img = Image.open(image_path).convert("RGB")
    x = clip_preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = torch.nn.functional.normalize(clip_model.encode_image(x), dim=1).float()
        # classifier dilatih di fitur ter-standardize; retrieval katalog tetap pakai
        # embedding L2-norm mentah (index katalog dibuat dari embedding mentah)
        probs = torch.softmax(classifier((feat - feat_mean) / feat_std), dim=1)[0].cpu().numpy()
    feat_np = feat.cpu().numpy()

    top3 = np.argsort(-probs)[:3]
    style_pred = [(idx_to_label[i], float(probs[i])) for i in top3]

    idxs, scores = nearest_neighbors(feat_np, cat_emb, top_k=top_k)
    verdict = confidence_verdict(scores[0])
    matches = []
    for gi, sim in zip(idxs[0], scores[0]):
        fn = str(cat_fnames[gi])
        row = meta.loc[fn] if fn in meta.index else None
        matches.append({
            "filename": fn, "similarity": float(sim),
            "artist_name": row["artist_name"] if row is not None else "?",
            "genre_name": row["genre_name"] if row is not None else "?",
            "style_name": row["style_name"] if row is not None else "?",
        })
    return {"input_image": img, "style_prediction": style_pred,
            "catalog_matches": matches, "verdict": verdict}


_VERDICT_TEXT = {
    "confirmed": "-> KEMUNGKINAN BESAR karya yang sama (skor tinggi & menonjol jelas dari kandidat lain)",
    "ambiguous": "-> TIDAK PASTI: beberapa karya mirip, TIDAK ADA yang menonjol jelas",
    "not_found": "-> kemungkinan besar karya ini TIDAK ADA di katalog kami",
}

def show_result(result, save_path=None):
    matches = result["catalog_matches"]
    verdict = result.get("verdict", "ambiguous")
    fig, axes = plt.subplots(1, 1 + len(matches), figsize=(3.2 * (1 + len(matches)), 4))

    axes[0].imshow(result["input_image"]); axes[0].axis("off")
    top_style, top_p = result["style_prediction"][0]
    axes[0].set_title(f"INPUT (v3 CLIP)\nprediksi: {top_style}\n({top_p*100:.0f}%)", fontsize=9, weight="bold")

    IMAGE_DIR = resolve_path(cfg["data"]["image_dir"])
    for i, m in enumerate(matches, start=1):
        mimg = Image.open(IMAGE_DIR / m["filename"]).convert("RGB")
        axes[i].imshow(mimg); axes[i].axis("off")
        is_the_match = (i == 1 and verdict == "confirmed")
        color = "green" if is_the_match else "black"
        tag = " (INI)" if is_the_match else ""
        axes[i].set_title(f"#{i} sim={m['similarity']:.2f}{tag}\n{m['artist_name']}\n{m['style_name']}",
                          fontsize=8, color=color)

    fig.suptitle("Scan lukisan (v3 CLIP) -> hasil pencarian katalog", fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path or resolve_path("outputs/v3_clip/demo_scan_result.png"), dpi=110)
    plt.show()

    print("=== PREDIKSI STYLE ===")
    for style, p in result["style_prediction"]:
        print(f"  {style:24s} {p*100:5.1f}%")
    print(f"\n=== KECOCOKAN KATALOG ({verdict.upper()}) ===")
    print(f"  {_VERDICT_TEXT[verdict]}")
    for i, m in enumerate(matches, 1):
        flag = "<-- INI (kemungkinan besar sama)" if (i == 1 and verdict == "confirmed") else ""
        print(f"  #{i} sim={m['similarity']:.3f} | {m['artist_name']:22s} | "
              f"{m['style_name']:22s} | {m['genre_name']:18s} | {m['filename']} {flag}")

## 4. Coba dengan 1 gambar

Ganti `IMAGE_PATH` -- dari dataset kita atau gambar dari luar (foto
internet/HP, path lokal apa saja).

In [ ]:
IMAGE_DIR = resolve_path(cfg["data"]["image_dir"])
IMAGE_PATH = IMAGE_DIR / "wikiart_00042.jpg"    # contoh dari dataset kita -- SAMA dgn demo v1/v2
# IMAGE_PATH = r"C:\Users\thori\Downloads\foto_lukisan_dari_hp.jpg"  # atau gambar dari luar

result = scan_lukisan(IMAGE_PATH, top_k=5)
show_result(result)

## 5. Bandingkan langsung ke hasil v1/v2 utk gambar yang SAMA

`IMAGE_PATH` di section 4 SENGAJA sama dgn contoh default di
`v1_11class_backup/03_demo_inference.ipynb` / `v2_22class/03_demo_inference.ipynb`
(`wikiart_00042.jpg`) -- jalankan ketiganya dgn gambar yang sama, lalu
bandingkan `style_prediction` & `catalog_matches` top-1 nya. Ini cara paling
konkret "melihat perbedaan antar model" di luar angka metrik, cocok utk
slide perbandingan live/screenshot di presentasi.

## 6. Catatan

- **Prediksi style** = classifier head kecil di atas embedding CLIP yang
  dibekukan -- akurasinya lihat `outputs/v3_clip/metrics_report.json`
  (section 6 `01_train.ipynb`), BUKAN hasil fine-tuning penuh seperti v1/v2.
- **Kecocokan katalog** = nearest-neighbor di embedding CLIP (512-d) --
  ruang vektor BEDA TOTAL dari v1/v2 (768-d, convnext_small), tidak bisa
  dicampur/dibandingkan similarity mentah secara langsung, hanya hasil
  akhirnya (gambar mana yang muncul top-1) yang relevan dibandingkan.
- Tidak ada demo webcam live di sini (lihat catatan scope di sel pertama).
- Ganti `IMAGE_PATH` di section 4 untuk coba gambar lain.